# Brute Force Detection — Cloudless SOC (VS Code)

**Goal:** Parse Windows-like auth logs, detect brute-force bursts, visualize them, and print simple alerts.

**How to run:** Use VS Code with the Python and Jupyter extensions installed. Activate your virtual env, then run all cells.

Dataset: `data/sample_windows_security.csv` (synthetic). Replace with your own to make it real-world.


In [ ]:
# Setup
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

# Display plots inline in VS Code/Jupyter
# (VS Code handles this automatically)

In [ ]:
# Load data
path = '../data/sample_windows_security.csv'
df = pd.read_csv(path, parse_dates=['timestamp'])
df.head()

In [ ]:
# Basic cleaning / typing
df['user'] = df['user'].astype(str)
df['src_ip'] = df['src_ip'].astype(str)
df['event_id'] = df['event_id'].astype(int)
df['status'] = df['status'].astype(str)

# Filter to failures (Windows 4625 = failed logon)
fails = df[(df['status'] == 'FAIL') & (df['event_id'] == 4625)].copy()
len(fails)

In [ ]:
# Aggregate failures over a rolling window
fails = fails.sort_values('timestamp')
fails.set_index('timestamp', inplace=True)

# Choose parameters you can tune
window = '5min'    # rolling horizon
threshold = 8      # alerts if >= threshold failures per window
group_cols = ['src_ip','user']

# Count failures by group per 5-min window
counts = fails.groupby(group_cols).resample(window).size().reset_index(name='failures')
counts.head()

In [ ]:
# Find suspicious windows (potential brute force)
sus = counts[counts['failures'] >= threshold].copy()
sus.sort_values(['failures'], ascending=False).head(10)

In [ ]:
# Visualize top offender (by total fails in suspicious windows)
if not sus.empty:
    top = sus.groupby('src_ip')['failures'].sum().sort_values(ascending=False).index[0]
    top_df = counts[counts['src_ip'] == top]
    top_user = top_df.groupby('user')['failures'].sum().sort_values(ascending=False).index[0]
    ax = top_df[top_df['user'] == top_user].plot(x='timestamp', y='failures', kind='line', title=f'Failures over time — {top} → {top_user}')
    plt.xlabel('Time')
    plt.ylabel('Failed logons')
    plt.show()
else:
    print("No suspicious windows found at current threshold.")

In [ ]:
# Alert printing
alerts = []
for _, row in sus.iterrows():
    alerts.append({
        'timestamp': row['timestamp'],
        'src_ip': row['src_ip'],
        'user': row['user'],
        'failures': int(row['failures']),
        'rule': f'BruteForce_{window}_{threshold}'
    })

print(f"Generated {len(alerts)} alerts")
alerts[:5]

## What to Document in Your Incident Report
- **Summary:** Suspected brute force from `src_ip` against `user`
- **Time window:** Which 5-min windows triggered
- **Thresholds & tuning:** Why you chose `threshold` and how you tried to reduce false positives
- **Next steps:** Blocklist IP, MFA/lockout policies, password hygiene, source geo/context